# Lab 3.2: Custom Slash Commands & Skills

**What you'll build:** A set of Claude Code commands and skills configurations -- and learn why context isolation (context: fork) matters for independent reviews.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- commands without structure, no context isolation | 5 min |
| 2 | The Right Way -- proper commands with $ARGUMENTS and skills with context: fork | 5 min |
| 3 | Your Turn -- design a command and skill system for a team workflow | 10 min |
| 4 | Stress Test -- validate your commands handle edge cases | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are setting up reusable workflows for a development team. The team needs:
1. A code review command that teammates can invoke on any file
2. A security audit skill that reviews code independently (not influenced by the current conversation)
3. Organized commands for testing (unit, integration, e2e)

We'll simulate command and skill files as Python data structures and use Claude to evaluate whether they are correctly configured.

In [ ]:
# The wrong way -- a single command that tries to do everything
WRONG_COMMANDS = {
    "location": "~/.claude/commands/",  # Personal, not shared!
    "files": {
        "review.md": {
            "content": """Review the code I'm working on. Check for bugs, security issues,
performance problems, and style issues. Also check if tests are adequate.
Be thorough but concise.""",
            "problems": [
                "No $ARGUMENTS -- can't specify what to review",
                "In user-level -- teammates can't use it",
                "Runs in current context -- biased by conversation",
                "Vague instructions -- 'be thorough but concise' is meaningless"
            ]
        }
    }
}

print("=== WRONG COMMAND SETUP ===")
print(f"Location: {WRONG_COMMANDS['location']} (shared via git: NO)")
for name, cmd in WRONG_COMMANDS['files'].items():
    print(f"\nCommand: /user:{name.replace('.md', '')}")
    print(f"Content preview: {cmd['content'][:80]}...")
    print(f"Problems:")
    for p in cmd['problems']:
        print(f"  - {p}")

---

## Phase 1: The Wrong Approach

Let's demonstrate why context contamination matters. We'll simulate a conversation where Claude generates code, then reviews its own code in the same context.

In [ ]:
# Simulate: Generate code, then review it in the SAME conversation (no fork)
BUGGY_CODE = '''
def process_payment(amount, currency="USD"):
    """Process a payment. Validates amount and converts currency."""
    # Bug 1: No validation for negative amounts
    # Bug 2: Currency conversion uses hardcoded rates (should be API call)
    # Bug 3: No error handling for conversion failures
    rates = {"USD": 1.0, "EUR": 0.85, "GBP": 0.73}
    converted = amount * rates[currency]  # KeyError if unknown currency
    return {"amount": converted, "status": "processed"}
'''

# Step 1: Pretend Claude just generated this code (set up the conversation context)
generation_context = [
    {"role": "user", "content": "Write a payment processing function."},
    {"role": "assistant", "content": f"Here's a payment processing function:\n```python\n{BUGGY_CODE}\n```\nThis handles payment processing with currency conversion."},
    {"role": "user", "content": "Now review that code for bugs and security issues. Return a JSON array of findings with 'location' and 'issue' fields. Return ONLY JSON."}
]

# Self-review (same context -- biased)
response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=generation_context
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
try:
    same_context_findings = json.loads(raw)
except json.JSONDecodeError:
    same_context_findings = [{"location": "parse_error", "issue": raw[:200]}]

print(f"=== SAME-CONTEXT REVIEW (no fork) ===")
print(f"Findings: {len(same_context_findings)}")
for f in same_context_findings:
    print(f"  - {f.get('location', '?')}: {f.get('issue', '?')}")

In [ ]:
# Step 2: Independent review (fresh context -- like context: fork)
independent_context = [
    {"role": "user", "content": f"""Review this code for bugs, security issues, and missing validation.

```python
{BUGGY_CODE}
```

Return a JSON array of findings. Each finding must have:
- "location": function or line description
- "issue": description of the problem
- "severity": "high", "medium", or "low"

Return ONLY the JSON array."""}
]

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=independent_context
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
try:
    forked_findings = json.loads(raw)
except json.JSONDecodeError:
    forked_findings = [{"location": "parse_error", "issue": raw[:200]}]

print(f"=== INDEPENDENT REVIEW (context: fork) ===")
print(f"Findings: {len(forked_findings)}")
for f in forked_findings:
    print(f"  - [{f.get('severity', '?')}] {f.get('location', '?')}: {f.get('issue', '?')}")

print(f"\n=== COMPARISON ===")
print(f"Same context (no fork): {len(same_context_findings)} findings")
print(f"Independent (fork):     {len(forked_findings)} findings")
if len(forked_findings) > len(same_context_findings):
    print("Independent review found more issues -- context isolation works.")

### Why context isolation matters

- **Same context = self-review bias.** The model retains its reasoning from generation, making it less likely to spot its own mistakes.
- **Forked context = fresh perspective.** A clean context forces the model to analyze the code without prior assumptions.
- **Skills with `context: fork`** provide this isolation automatically in Claude Code.

---

## Phase 2: The Right Approach

Let's build properly structured commands and skills.

In [ ]:
# The right way -- organized commands and skills
RIGHT_SETUP = {
    "commands": {
        "location": ".claude/commands/",  # Project-level, shared via git
        "files": {
            "review.md": {
                "content": """Review the file at $ARGUMENTS for:
1. Logic errors and bugs
2. Missing input validation
3. Error handling gaps

DO flag: code that will crash or produce wrong results.
DO NOT flag: style preferences or minor naming issues.

Output findings as a numbered list with severity (high/medium/low).""",
                "invocation": "/project:review src/api/users.py"
            },
            "test/unit.md": {
                "content": """Generate unit tests for $ARGUMENTS.

Requirements:
- Use pytest with describe/it pattern
- Mock all external dependencies
- Test: happy path, edge cases (empty, null, boundary), error cases
- Name: test_<behavior>_<condition>_<expected>
- One assertion per test""",
                "invocation": "/project:test:unit src/api/users.py"
            },
            "test/integration.md": {
                "content": """Generate integration tests for $ARGUMENTS.

Requirements:
- Use pytest with real database (test container)
- Test full request/response cycle
- Verify database state after operations
- Clean up test data in teardown""",
                "invocation": "/project:test:integration src/api/users.py"
            }
        }
    },
    "skills": {
        "location": ".claude/skills/",
        "files": {
            "security-audit/SKILL.md": {
                "frontmatter": {
                    "name": "security-audit",
                    "description": "Independent security review of code",
                    "context": "fork"  # KEY: isolated context
                },
                "content": """Perform a security audit of the specified code.

Focus areas:
1. Input validation and sanitization
2. Authentication and authorization checks
3. SQL injection and XSS vulnerabilities
4. Sensitive data exposure (logging, error messages)
5. Cryptographic weaknesses

For each finding, provide:
- Severity (critical/high/medium/low)
- Specific code location
- Recommended fix"""
            }
        }
    }
}

print("=== CORRECT SETUP ===")
print(f"\nCommands ({RIGHT_SETUP['commands']['location']}):")
for name, cmd in RIGHT_SETUP['commands']['files'].items():
    print(f"  /project:{name.replace('.md', '').replace('/', ':')}")
    print(f"    Invocation: {cmd['invocation']}")

print(f"\nSkills ({RIGHT_SETUP['skills']['location']}):")
for name, skill in RIGHT_SETUP['skills']['files'].items():
    fm = skill['frontmatter']
    print(f"  {fm['name']}: {fm['description']}")
    print(f"    context: {fm['context']} (isolated from conversation)")

---

## Phase 3: Your Turn

Design a command and skill system for a team that needs:
1. A **documentation generator** command (takes a file path, generates docs)
2. A **refactoring assistant** skill (needs isolated context to avoid bias from prior discussion)
3. **Test commands** organized by type (unit, integration, e2e)
4. All commands must be shared with the team

In [ ]:
# =============================================================
# TODO: Design your command and skill system
# =============================================================

YOUR_SETUP = {
    "commands": {
        "location": ".claude/commands/",  # Must be project-level!
        "files": {
            # TODO: Add your command files
            # "docs.md": {
            #     "content": "...",
            #     "invocation": "/project:docs src/api/users.py"
            # },
        }
    },
    "skills": {
        "location": ".claude/skills/",
        "files": {
            # TODO: Add your skill files
            # "refactor/SKILL.md": {
            #     "frontmatter": {"name": "...", "description": "...", "context": "fork"},
            #     "content": "..."
            # },
        }
    }
}

print("Your setup:")
print(json.dumps(YOUR_SETUP, indent=2))

In [ ]:
# =============================================================
# Checker: validates your command/skill design
# =============================================================

def check_commands_skills(setup):
    errors = []
    
    # Check 1: Commands must be project-level
    cmd_loc = setup.get('commands', {}).get('location', '')
    if '~' in cmd_loc:
        errors.append("Commands are in user-level -- move to .claude/commands/ for team sharing.")
    
    # Check 2: Must have at least one command
    cmd_files = setup.get('commands', {}).get('files', {})
    if not cmd_files:
        errors.append("No command files defined.")
    
    # Check 3: Commands should use $ARGUMENTS
    for name, cmd in cmd_files.items():
        content = cmd.get('content', '')
        if '$ARGUMENTS' not in content:
            errors.append(f"Command '{name}' missing $ARGUMENTS placeholder.")
    
    # Check 4: Must have at least one skill with context: fork
    skill_files = setup.get('skills', {}).get('files', {})
    has_forked_skill = False
    for name, skill in skill_files.items():
        fm = skill.get('frontmatter', {})
        if fm.get('context') == 'fork':
            has_forked_skill = True
    
    if not has_forked_skill:
        errors.append("No skill with context: fork found. The refactoring assistant needs isolated context.")
    
    # Check 5: Test commands should be nested
    test_commands = [n for n in cmd_files if 'test' in n.lower()]
    if not test_commands:
        errors.append("No test commands found. Add organized test commands (unit, integration, e2e).")
    
    # Results
    print("=== COMMAND/SKILL VALIDATION ===")
    if not errors:
        print("PASSED -- Well-designed command and skill system!")
        print(f"  Commands: {len(cmd_files)}")
        print(f"  Skills: {len(skill_files)}")
        print(f"  Has forked skill: {has_forked_skill}")
    else:
        for e in errors:
            print(f"  [ERROR] {e}")
        print(f"\nFix {len(errors)} error(s) and try again.")
    
    return len(errors) == 0

check_commands_skills(YOUR_SETUP)

---

## Phase 4: Stress Test

Verify that your commands produce consistent, useful output across different inputs.

In [ ]:
# Test: simulate invoking your documentation command with the payment code
doc_commands = {n: c for n, c in YOUR_SETUP['commands']['files'].items() 
                if 'doc' in n.lower()}

if doc_commands:
    cmd_name, cmd_data = next(iter(doc_commands.items()))
    # Simulate $ARGUMENTS replacement
    prompt = cmd_data['content'].replace('$ARGUMENTS', f'the following code:\n```python\n{BUGGY_CODE}\n```')
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(f"=== Documentation Command Output ===")
    print(response.content[0].text[:500])
    print("\n... (truncated)")
else:
    print("No documentation command found -- add one and rerun.")

# Test: verify forked skill has all required frontmatter
print("\n=== Skill Frontmatter Validation ===")
for name, skill in YOUR_SETUP['skills']['files'].items():
    fm = skill.get('frontmatter', {})
    required = ['name', 'description', 'context']
    missing = [r for r in required if r not in fm]
    if missing:
        print(f"  {name}: MISSING fields {missing}")
    else:
        print(f"  {name}: All required frontmatter present")
        print(f"    name: {fm['name']}")
        print(f"    context: {fm['context']}")

### Key Takeaways

1. **Commands are simple prompt templates** in `.claude/commands/`. Use `$ARGUMENTS` for input and nested directories for organization.
2. **Skills provide context isolation** via `context: fork`. Use them for review tasks that need a fresh perspective.
3. **Project-level commands** (`.claude/commands/`) are shared via git. User-level (`~/.claude/commands/`) are personal.
4. **Nested directories create namespaced commands.** `.claude/commands/test/unit.md` becomes `/project:test:unit`.
5. **Same-context review is biased.** The model is less likely to find its own mistakes. Fork solves this.